# CALIPSO Downloader (interactive notebook)

This notebook provides an interactive interface to build download tasks for CALIPSO (L0-L3) products and run the low-level bash downloader `download_file.sh`. It supports: direct URL lists, URL templates with date expansion, and granule filename lists.

Notes:
- Keep credentials out of the notebook. Use `~/.netrc` or environment variables `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD`.
- Raw files are kept as-is by default.


In [ ]:
import subprocess
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
from typing import List

def load_urls_from_file(path: str) -> List[str]:
    with open(path, 'r') as fh:
        return [line.strip() for line in fh if line.strip() and not line.startswith('#')]

def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days) + 1):
        yield start_date + timedelta(n)

def expand_template(template: str, product: str = None, start: str = None, end: str = None, granules: List[str] = None) -> List[str]:
    urls = []
    if granules:
        if start and end:
            s = datetime.strptime(start, '%Y-%m-%d')
            e = datetime.strptime(end, '%Y-%m-%d')
            for dt in daterange(s, e):
                for g in granules:
                    url = template.format(PRODUCT=product or '', YYYY=dt.strftime('%Y'), MM=dt.strftime('%m'), DD=dt.strftime('%d'), YYYYMMDD=dt.strftime('%Y%m%d'), FNAME=g)
                    urls.append(url)
        else:
            for g in granules:
                url = template.format(PRODUCT=product or '', YYYY='', MM='', DD='', YYYYMMDD='', FNAME=g)
                urls.append(url)
    else:
        if not start or not end:
            raise ValueError('start and end required when not using granule-list')
        s = datetime.strptime(start, '%Y-%m-%d')
        e = datetime.strptime(end, '%Y-%m-%d')
        for dt in daterange(s, e):
            url = template.format(PRODUCT=product or '', YYYY=dt.strftime('%Y'), MM=dt.strftime('%m'), DD=dt.strftime('%d'), YYYYMMDD=dt.strftime('%Y%m%d'))
            urls.append(url)
    return urls


In [ ]:
def build_download_cmd(url: str, outdir: str, netrc_file: str = None, username: str = None, password: str = None) -> List[str]:
    script = Path('download_file.sh').resolve()  # assume notebook runs in obs_downloader/ or adjust path accordingly
    cmd = [str(script), url, outdir]
    if netrc_file:
        cmd += ['--netrc-file', netrc_file]
    if username and password:
        cmd += ['--username', username, '--password', password]
    return cmd

def run_download(url: str, outdir: str, netrc_file: str = None, username: str = None, password: str = None) -> subprocess.CompletedProcess:
    cmd = build_download_cmd(url, outdir, netrc_file=netrc_file, username=username, password=password)
    print('Running:', ' '.join(cmd))
    return subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def download_urls(urls: List[str], outdir: str, concurrency: int = 4, netrc_file: str = None, username: str = None, password: str = None, dry_run: bool = False):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    if dry_run:
        for u in urls:
            print('DRY:', ' '.join(build_download_cmd(u, str(outdir), netrc_file=netrc_file, username=username, password=password)))
        return

    results = []
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futures = {ex.submit(run_download, u, str(outdir), netrc_file, username, password): u for u in urls}
        for fut in as_completed(futures):
            u = futures[fut]
            try:
                res = fut.result()
                if res.returncode == 0:
                    print(f'DONE: {u}')
                else:
                    print(f'FAILED: {u} -> returncode={res.returncode}\n{res.stderr}')
                results.append((u, res.returncode, res.stdout, res.stderr))
            except Exception as e:
                print('EXC for', u, e)
                results.append((u, -1, '', str(e)))
    return results


In [ ]:
# Example usage - adjust variables below
OUTDIR = 'downloads'
CONCURRENCY = 4
NETRC_FILE = None  # e.g. '/Users/you/.netrc'
USERNAME = os.environ.get('EARTHDATA_USERNAME')
PASSWORD = os.environ.get('EARTHDATA_PASSWORD')

# Option A: load direct URLs from a file
# urls = load_urls_from_file('urls.txt')

# Option B: expand a template over a date range (and optionally a granule list)
TEMPLATE = 'https://example.org/data/{PRODUCT}/{YYYY}/{MM}/{DD}/{FNAME}'
PRODUCT = 'CAL_LID_L1-Standard-V4-10'
START = '2021-01-01'
END = '2021-01-03'
GRANULES = ['CAL_LID_L1-Standard-V4-10-A12345.hdf']  # or load from a file

urls = expand_template(TEMPLATE, product=PRODUCT, start=START, end=END, granules=GRANULES)
print('Planned downloads:', len(urls))

# Dry run first
download_urls(urls, OUTDIR, concurrency=CONCURRENCY, netrc_file=NETRC_FILE, username=USERNAME, password=PASSWORD, dry_run=True)

# When ready, run for real (uncomment)
# results = download_urls(urls, OUTDIR, concurrency=CONCURRENCY, netrc_file=NETRC_FILE, username=USERNAME, password=PASSWORD, dry_run=False)
# print(results)


Notes on credentials:

- Create a `~/.netrc` entry for Earthdata if you prefer not to store credentials in environment variables. Example:
```text
machine urs.earthdata.nasa.gov
login YOUR_USERNAME
password YOUR_PASSWORD
```
- Or set `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` in your environment before launching the notebook.
